In [3]:
import pandas as pd
from pathlib import Path

# Put kepler_stellar_17.csv.gz in the same folder as this notebook.
FILE = Path("/kaggle/input/datasets/sairakeshnagaram/koi-stellars1/kepler_stellar_17.csv")

df = pd.read_csv(FILE)  # No compression="gzip"

# Confirm that this is the expected Kepler DR25 catalog.
print("Rows:", len(df))
print("Unique Kepler IDs:", df["kepid"].nunique())
print("Catalog versions:")
print(df["st_delivname"].value_counts(dropna=False))

# Check the required columns before filtering.
required = ["kepid", "teff", "dataspan", "dutycycle"]
missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Convert relevant columns to numeric.
for col in ["teff", "dataspan", "dutycycle"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Count stars at each stage so the selection is transparent.
stage_0 = df.drop_duplicates("kepid")
stage_1 = stage_0[
    stage_0["teff"].between(5300, 6000, inclusive="both")
]
stage_2 = stage_1[
    stage_1["dataspan"] >= 730
]
stage_3 = stage_2[
    stage_2["dutycycle"] >= 0.60
]

counts = pd.DataFrame({
    "Selection stage": [
        "All unique DR25 stars",
        "G-star temperature range (5300–6000 K)",
        "At least 730 days of data span",
        "Duty cycle at least 0.60"
    ],
    "Number of stars": [
        len(stage_0),
        len(stage_1),
        len(stage_2),
        len(stage_3)
    ]
})

print("\nPRELIMINARY G-STAR DENOMINATOR")
display(counts)

print("\nTemperature range:", stage_3["teff"].min(),
      "to", stage_3["teff"].max(), "K")
print("Missing duty-cycle values in original catalog:",
      df["dutycycle"].isna().sum())

# Save the preliminary selected target-star list.
stage_3.to_csv("preliminary_G_star_targets.csv", index=False)
print("\nSaved preliminary_G_star_targets.csv")


Rows: 200038
Unique Kepler IDs: 200038
Catalog versions:
st_delivname
q1_q17_dr25_stellar    200038
Name: count, dtype: int64

PRELIMINARY G-STAR DENOMINATOR


,Selection stage,Number of stars
0,All unique DR25 stars,200038
1,G-star temperature range (5300–6000 K),72581
2,At least 730 days of data span,66797
3,Duty cycle at least 0.60,64575



Temperature range: 5300 to 6000 K
Missing duty-cycle values in original catalog: 1336

Saved preliminary_G_star_targets.csv
